# Core PCE Shock Volatility

This standalone notebook complements—but does not alter—the Core PCE Inflation Shock Momentum index. ISM measures the **direction and breadth** of persistent surprises; this notebook measures the **magnitude** of category-level inflation surprises regardless of sign.

For category residuals $e_{i,t}$ and current expenditure weights $w_{i,t}$, the monthly total shock variance is

$$Q_t=\sum_i w_{i,t}e_{i,t}^2.
$$

It decomposes exactly into a squared aggregate/common shock and cross-category dispersion:

$$Q_t=\left(\sum_i w_{i,t}e_{i,t}\right)^2+\sum_i w_{i,t}(e_{i,t}-\bar e_t)^2.
$$

The headline volatility series applies an exponentially weighted moving average (EWMA) to these variances and then takes square roots. A standardized version divides each residual by its category's contemporaneous rolling AR residual scale. These are ex-post realized shock measures, not survey- or market-implied uncertainty.

## 1. Settings and imports

The baseline mirrors the ISM replication: a 117-category core basket, simple monthly inflation, rolling 120-month AR(1) models with a constant, current nominal-spending weights, and available-basket renormalization. The EWMA half-life is 12 months. Residuals are monthly percentage-point surprises and are not annualized.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd().resolve()
CORE_LINES_PATH = ROOT / 'core_lines.xlsx'
CACHE = ROOT / 'cache'
OUT = ROOT / 'output' / 'core_pce_shock_volatility'
CACHE.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

BEA_DATASET = 'NIUnderlyingDetail'
BEA_FREQUENCY = 'M'
PRICE_TABLE = 'U20404'
SPEND_TABLE = 'U20405'
START_YEAR = 1959
END_YEAR = datetime.now().year
YEAR_CHUNKS = [(1959, 1999), (2000, END_YEAR)]

WINDOW = 120
AR_LAGS = 1
MIN_LINES = 50
EWMA_HALFLIFE = 12
EWMA_MIN_PERIODS = 12
TRAILING_WINDOW = 12
REFRESH = False
TAG = f'ar{AR_LAGS}_hl{EWMA_HALFLIFE}'

INDEX_CSV = OUT / f'core_pce_shock_volatility_{TAG}.csv'
SHOCKS_CSV = OUT / f'core_pce_category_shocks_{TAG}.csv'
FIGURE_PNG = OUT / f'core_pce_shock_volatility_{TAG}.png'

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 170)
warnings.filterwarnings('default')
try:
    display
except NameError:
    display = print

print({'window': WINDOW, 'ar_lags': AR_LAGS, 'ewma_half_life': EWMA_HALFLIFE, 'min_lines': MIN_LINES, 'refresh': REFRESH})

## 2. BEA data helpers

Cached BEA Underlying Detail tables are used by default. If `REFRESH=True`, the credential is read from the `BEA_API_KEY` environment variable or the local `API Keys.txt` file. It is never printed or exported.

In [ ]:
BEA_URL = 'https://apps.bea.gov/api/data'
MISSING_VALUES = {'', '.', '-', '--', 'NA', '(NA)', 'N/A', 'nan', 'None'}

def read_bea_key():
    key = os.environ.get('BEA_API_KEY', '').strip()
    if key:
        return key
    key_file = ROOT / 'API Keys.txt'
    if key_file.exists():
        match = re.search(r'(?im)^\s*BEA:\s*(.+?)\s*$', key_file.read_text(encoding='utf-8'))
        if match:
            return match.group(1).strip()
    raise RuntimeError('BEA credential not found. Set BEA_API_KEY or add a BEA: entry to API Keys.txt.')

def parse_bea_month(value):
    match = re.fullmatch(r'(\d{4})M(\d{1,2})', str(value).strip())
    if not match:
        raise ValueError(f'Unexpected BEA monthly period: {value!r}')
    return pd.Timestamp(int(match.group(1)), int(match.group(2)), 1)

def as_float(value):
    text = str(value).replace(',', '').strip()
    return np.nan if text in MISSING_VALUES else float(text)

def unwrap_results(raw):
    results = raw.get('BEAAPI', {}).get('Results', {})
    if isinstance(results, list):
        candidates = [item for item in results if isinstance(item, dict) and ('Data' in item or 'Error' in item)]
        results = candidates[-1] if candidates else {}
    if not isinstance(results, dict):
        raise RuntimeError('Unexpected BEA Results payload')
    if 'Error' in results:
        error = results['Error']
        if isinstance(error, list):
            error = error[0] if error else {}
        code = error.get('APIErrorCode', 'unknown') if isinstance(error, dict) else 'unknown'
        message = error.get('APIErrorDescription', str(error)) if isinstance(error, dict) else str(error)
        raise RuntimeError(f'BEA API error {code}: {message}')
    return results

def normalize_bea_rows(rows):
    records = []
    for row in rows or []:
        if row.get('LineNumber') is None or row.get('TimePeriod') is None:
            continue
        records.append({
            'line': int(str(row['LineNumber']).strip()),
            'description': str(row.get('LineDescription', '')).strip(),
            'series_code': str(row.get('SeriesCode', '')).strip(),
            'date': parse_bea_month(row['TimePeriod']),
            'value': as_float(row.get('DataValue')),
        })
    return pd.DataFrame(records, columns=['line', 'description', 'series_code', 'date', 'value'])

def fetch_bea(table, year0, year1, retries=4):
    params = {
        'UserID': read_bea_key(),
        'method': 'GetData',
        'DataSetName': BEA_DATASET,
        'TableName': table,
        'Frequency': BEA_FREQUENCY,
        'Year': ','.join(str(year) for year in range(year0, year1 + 1)),
        'ResultFormat': 'JSON',
    }
    body = urlencode(params).encode('utf-8')
    last_error = None
    for attempt in range(retries):
        try:
            request = Request(BEA_URL, data=body, headers={'User-Agent': 'core-pce-shock-volatility/1.0'})
            with urlopen(request, timeout=180) as response:
                raw = json.loads(response.read().decode('utf-8'))
            data = normalize_bea_rows(unwrap_results(raw).get('Data', []))
            if data.empty:
                raise RuntimeError(f'BEA returned no data for {table}, {year0}-{year1}')
            return data
        except (HTTPError, URLError, TimeoutError, RuntimeError, ValueError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(20 * (attempt + 1))
    raise RuntimeError(f'Could not fetch {table}, {year0}-{year1}: {last_error}')

def cache_path(table):
    return CACHE / f'bea_{BEA_DATASET}_{table}_{BEA_FREQUENCY}_{START_YEAR}_{END_YEAR}.csv'

def bea_table(table, refresh=REFRESH):
    path = cache_path(table)
    if path.exists() and not refresh:
        cached = pd.read_csv(path, parse_dates=['date'])
        required = {'line', 'description', 'series_code', 'date', 'value'}
        if required.issubset(cached.columns):
            return cached.sort_values(['line', 'date']).reset_index(drop=True)
        warnings.warn(f'Ignoring malformed cache: {path}')
    chunks = []
    for chunk_no, (year0, year1) in enumerate(YEAR_CHUNKS):
        chunks.append(fetch_bea(table, year0, year1))
        if chunk_no < len(YEAR_CHUNKS) - 1:
            time.sleep(2)
    data = pd.concat(chunks, ignore_index=True).sort_values(['line', 'date']).reset_index(drop=True)
    if data.duplicated(['line', 'date']).any():
        raise AssertionError(f'Duplicate BEA line/date rows in {table}')
    data.to_csv(path, index=False, date_format='%Y-%m-%d')
    return data

def pivot_selected(data, lines, table):
    selected = data[data['line'].isin(lines)].copy()
    if selected.duplicated(['line', 'date']).any():
        raise AssertionError(f'Duplicate selected line/date rows in {table}')
    return selected.pivot(index='date', columns='line', values='value').sort_index().reindex(columns=lines)

## 3. Core basket and data coverage

The configured 117 lines must be unique and present in both BEA tables. Description drift, missing observations, nonpositive spending, and partial-basket months are made visible rather than silently ignored.

In [ ]:
run_started = time.perf_counter()
raw_lines = pd.read_excel(CORE_LINES_PATH)
required_columns = {'Line Item', 'Description'}
if not required_columns.issubset(raw_lines.columns):
    raise ValueError(f'{CORE_LINES_PATH.name} must contain {sorted(required_columns)}')

core_lines = raw_lines.rename(columns={'Line Item': 'line', 'Description': 'description', 'Code': 'code'}).copy()
core_lines['line'] = core_lines['line'].astype(int)
core_lines['description'] = core_lines['description'].astype(str).str.strip()
if 'code' not in core_lines:
    core_lines['code'] = ''
core_lines['code'] = core_lines['code'].fillna('').astype(str).str.strip()
core_lines = core_lines[['line', 'description', 'code']].sort_values('line').reset_index(drop=True)
assert len(core_lines) == 117, f'Expected 117 core lines, found {len(core_lines)}'
assert core_lines['line'].is_unique, 'Core line numbers must be unique'
assert core_lines['description'].is_unique, 'Core descriptions must be unique'
LINES = core_lines['line'].tolist()
LINE_DESCRIPTION = core_lines.set_index('line')['description']

price_long = bea_table(PRICE_TABLE)
spend_long = bea_table(SPEND_TABLE)
for table_name, data in [(PRICE_TABLE, price_long), (SPEND_TABLE, spend_long)]:
    missing_lines = sorted(set(LINES) - set(data['line'].unique()))
    if missing_lines:
        raise AssertionError(f'{table_name} is missing configured lines: {missing_lines}')

prices = pivot_selected(price_long, LINES, PRICE_TABLE)
spend = pivot_selected(spend_long, LINES, SPEND_TABLE)
full_dates = pd.date_range(min(prices.index.min(), spend.index.min()), max(prices.index.max(), spend.index.max()), freq='MS')
prices = prices.reindex(full_dates).where(lambda frame: frame > 0)
spend = spend.reindex(full_dates)

def normalized_description(value):
    return re.sub(r'\s+', ' ', str(value).strip()).casefold()

description_checks = []
for table_name, data in [(PRICE_TABLE, price_long), (SPEND_TABLE, spend_long)]:
    observed_descriptions = data[data['line'].isin(LINES)].drop_duplicates('line').set_index('line')['description']
    for line in LINES:
        expected = LINE_DESCRIPTION.loc[line]
        observed = observed_descriptions.get(line, '')
        if normalized_description(expected) != normalized_description(observed):
            description_checks.append({'table': table_name, 'line': line, 'workbook': expected, 'bea': observed})
description_mismatches = pd.DataFrame(description_checks)
if not description_mismatches.empty:
    warnings.warn(f'{len(description_mismatches)} workbook/BEA description mismatches; inspect description_mismatches.')
    display(description_mismatches)

nonpositive_mask = spend.notna() & spend.le(0)
nonpositive_spend = int(nonpositive_mask.sum().sum())
if nonpositive_spend:
    warnings.warn(f'Found {nonpositive_spend:,} nonpositive spending observations; they will not receive weights.')

print(f'Loaded {len(LINES)} core lines; BEA data span {full_dates.min():%Y-%m} to {full_dates.max():%Y-%m}.')
print(f'Description mismatches: {len(description_mismatches)}; nonpositive spending observations: {nonpositive_spend}.')
display(core_lines.head(12))

## 4. Monthly inflation and rolling AR shocks

Monthly category inflation is the simple percentage change in U20404. For every end month $t$, an AR($p$) with a constant is estimated on the 120-month inflation window ending at $t$. The current in-sample residual is the category shock. Its scale is the regression RMSE from the same window, so the standardized residual is a contemporaneous historical reconstruction rather than a strictly ex-ante forecast error.

In [ ]:
infl = prices.pct_change(fill_method=None) * 100.0

def rolling_endpoint_shock(y, window=WINDOW, p=AR_LAGS):
    y = np.asarray(y, dtype=float)
    if window <= p + 1:
        raise ValueError('The rolling window must exceed the AR parameter count.')
    endpoint = np.full(len(y), np.nan)
    residual_scale = np.full(len(y), np.nan)
    coefficients = np.full((len(y), p + 1), np.nan)
    for end in range(window - 1, len(y)):
        sample = y[end - window + 1:end + 1]
        if np.isnan(sample).any():
            continue
        dependent = sample[p:]
        design = np.column_stack([np.ones(window - p)] + [sample[p-lag:window-lag] for lag in range(1, p + 1)])
        beta = np.linalg.lstsq(design, dependent, rcond=None)[0]
        residual = dependent - design @ beta
        dof = len(dependent) - len(beta)
        endpoint[end] = residual[-1]
        residual_scale[end] = np.sqrt(residual @ residual / dof) if dof > 0 else np.nan
        coefficients[end] = beta
    return endpoint, residual_scale, coefficients

residual = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
residual_scale = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
intercept = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
ar_coefficients = {}
for line in LINES:
    endpoint, scale, coefficients = rolling_endpoint_shock(infl[line].to_numpy())
    residual[line] = endpoint
    residual_scale[line] = scale
    intercept[line] = coefficients[:, 0]
    ar_coefficients[line] = coefficients

standardized_residual = residual.div(residual_scale).where(residual_scale.gt(0))
first_residual_dates = residual.apply(lambda s: s.first_valid_index())
full_history_lines = [line for line in LINES if prices[line].first_valid_index() == pd.Timestamp(START_YEAR, 1, 1)]
if full_history_lines:
    expected_first = pd.Timestamp(1969, 1, 1)
    unexpected = first_residual_dates.loc[full_history_lines][first_residual_dates.loc[full_history_lines] != expected_first]
    if not unexpected.empty:
        raise AssertionError(f'Unexpected first rolling residual dates: {unexpected.to_dict()}')

check_line = next(line for line in LINES if residual[line].notna().any())
check_date = residual[check_line].first_valid_index()
check_loc = infl.index.get_loc(check_date)
check_sample = infl[check_line].iloc[check_loc-WINDOW+1:check_loc+1].to_numpy()
check_y = check_sample[AR_LAGS:]
check_x = np.column_stack([np.ones(WINDOW-AR_LAGS)] + [check_sample[AR_LAGS-lag:WINDOW-lag] for lag in range(1, AR_LAGS+1)])
check_beta = np.linalg.lstsq(check_x, check_y, rcond=None)[0]
check_endpoint = check_y[-1] - check_x[-1] @ check_beta
np.testing.assert_allclose(residual.loc[check_date, check_line], check_endpoint, atol=1e-12, rtol=0)
print(f'First full-history endpoint residual: {pd.Timestamp(1969, 1, 1):%Y-%m}')
print(f'Direct one-window residual check passed for line {check_line} at {check_date:%Y-%m}.')

## 5. Current weights and monthly shock decomposition

Raw shocks receive current-month spending weights when the residual is valid and spending is finite and positive. Weights are renormalized over the available categories and a month is published when at least `MIN_LINES` remain. Standardized shocks use an independently renormalized available basket if a residual scale is unavailable.

In [ ]:
spend = spend.reindex(index=infl.index, columns=LINES)
raw_valid = residual.notna() & spend.notna() & spend.gt(0)
weighted_spend = spend.where(raw_valid)
available_spend = weighted_spend.sum(axis=1, min_count=1)
full_core_spend = spend.where(spend.gt(0)).sum(axis=1, min_count=1)
weights = weighted_spend.div(available_spend, axis=0)
n_valid = raw_valid.sum(axis=1).astype(int)
weight_coverage = available_spend.div(full_core_spend)
eligible = n_valid.ge(MIN_LINES) & available_spend.gt(0)

aggregate_shock = (weights * residual).sum(axis=1, min_count=1).where(eligible)
total_variance_current = (weights * residual.pow(2)).sum(axis=1, min_count=1).where(eligible)
aggregate_variance_current = aggregate_shock.pow(2)
centered_shock = residual.sub(aggregate_shock, axis=0)
dispersion_variance_current = (weights * centered_shock.pow(2)).sum(axis=1, min_count=1).where(eligible)
shock_intensity = np.sqrt(total_variance_current.clip(lower=0))

standardized_valid = standardized_residual.notna() & spend.notna() & spend.gt(0)
standardized_spend = spend.where(standardized_valid)
standardized_available_spend = standardized_spend.sum(axis=1, min_count=1)
standardized_weights = standardized_spend.div(standardized_available_spend, axis=0)
standardized_n_valid = standardized_valid.sum(axis=1).astype(int)
standardized_weight_coverage = standardized_available_spend.div(full_core_spend)
standardized_eligible = standardized_n_valid.ge(MIN_LINES) & standardized_available_spend.gt(0)
standardized_total_variance_current = (standardized_weights * standardized_residual.pow(2)).sum(axis=1, min_count=1).where(standardized_eligible)
standardized_shock_intensity = np.sqrt(standardized_total_variance_current.clip(lower=0))

published_weight_sum = weights.sum(axis=1, min_count=1).where(eligible).dropna()
np.testing.assert_allclose(published_weight_sum.to_numpy(), 1.0, atol=1e-12, rtol=0)
decomposition_error = (total_variance_current - aggregate_variance_current - dispersion_variance_current).dropna()
np.testing.assert_allclose(decomposition_error.to_numpy(), 0.0, atol=1e-12, rtol=0)
assert total_variance_current.dropna().ge(-1e-14).all()
assert aggregate_variance_current.dropna().ge(-1e-14).all()
assert dispersion_variance_current.dropna().ge(-1e-14).all()

published_dates = total_variance_current.dropna().index
partial_months = weight_coverage.loc[published_dates].lt(1 - 1e-12)
if partial_months.any():
    warnings.warn(f'{int(partial_months.sum())} published months use a partial dynamically renormalized basket.')
print(f'Raw shock span: {published_dates.min():%Y-%m} to {published_dates.max():%Y-%m}.')
print(f'Minimum line count: {n_valid.loc[published_dates].min()} of {len(LINES)}; minimum spending coverage: {weight_coverage.loc[published_dates].min():.2%}.')
print(f'Maximum absolute monthly decomposition error: {decomposition_error.abs().max():.3e}.')

## 6. Volatility smoothing

Volatility is the square root of smoothed variance. Smoothing each variance component with the same linear EWMA preserves the decomposition in variance space. The component standard deviations do not add; their **squares** do. A 12-month trailing RMS is also retained as a transparent robustness series.

In [ ]:
def ewma_variance(series):
    return series.ewm(halflife=EWMA_HALFLIFE, adjust=False, min_periods=EWMA_MIN_PERIODS, ignore_na=True).mean()

total_variance_ewma = ewma_variance(total_variance_current)
aggregate_variance_ewma = ewma_variance(aggregate_variance_current)
dispersion_variance_ewma = ewma_variance(dispersion_variance_current)
standardized_total_variance_ewma = ewma_variance(standardized_total_variance_current)

shock_volatility_ewma = np.sqrt(total_variance_ewma.clip(lower=0))
aggregate_volatility_ewma = np.sqrt(aggregate_variance_ewma.clip(lower=0))
dispersion_volatility_ewma = np.sqrt(dispersion_variance_ewma.clip(lower=0))
standardized_shock_volatility_ewma = np.sqrt(standardized_total_variance_ewma.clip(lower=0))
shock_volatility_trailing12 = np.sqrt(total_variance_current.rolling(TRAILING_WINDOW, min_periods=TRAILING_WINDOW).mean().clip(lower=0))
common_variance_share = aggregate_variance_ewma.div(total_variance_ewma).where(total_variance_ewma.gt(0)).clip(0, 1)

smoothed_decomposition_error = (total_variance_ewma - aggregate_variance_ewma - dispersion_variance_ewma).dropna()
np.testing.assert_allclose(smoothed_decomposition_error.to_numpy(), 0.0, atol=1e-12, rtol=0)
assert common_variance_share.dropna().between(0, 1).all()

volatility_output = pd.DataFrame({
    'date': infl.index,
    'aggregate_shock': aggregate_shock.to_numpy(),
    'shock_intensity': shock_intensity.to_numpy(),
    'total_variance_current': total_variance_current.to_numpy(),
    'aggregate_variance_current': aggregate_variance_current.to_numpy(),
    'dispersion_variance_current': dispersion_variance_current.to_numpy(),
    'shock_volatility_ewma': shock_volatility_ewma.to_numpy(),
    'aggregate_volatility_ewma': aggregate_volatility_ewma.to_numpy(),
    'dispersion_volatility_ewma': dispersion_volatility_ewma.to_numpy(),
    'shock_volatility_trailing12': shock_volatility_trailing12.to_numpy(),
    'standardized_shock_intensity': standardized_shock_intensity.to_numpy(),
    'standardized_shock_volatility_ewma': standardized_shock_volatility_ewma.to_numpy(),
    'common_variance_share': common_variance_share.to_numpy(),
    'n_valid': n_valid.to_numpy(),
    'weight_coverage': weight_coverage.to_numpy(),
    'standardized_n_valid': standardized_n_valid.to_numpy(),
    'standardized_weight_coverage': standardized_weight_coverage.to_numpy(),
}).dropna(subset=['total_variance_current']).reset_index(drop=True)

latest_smoothed = volatility_output.dropna(subset=['shock_volatility_ewma']).iloc[-1]
print(f"First smoothed volatility: {volatility_output.dropna(subset=['shock_volatility_ewma'])['date'].iloc[0]:%Y-%m}")
print(f"Latest total volatility ({latest_smoothed['date']:%Y-%m}): {latest_smoothed['shock_volatility_ewma']:.4f} monthly percentage points.")
print(f"Latest aggregate component: {latest_smoothed['aggregate_volatility_ewma']:.4f}; dispersion component: {latest_smoothed['dispersion_volatility_ewma']:.4f}.")
display(volatility_output.tail().set_index('date').round(4))

## 7. Visualization

Panel A shows why smoothing is necessary. Panel B decomposes the headline volatility in variance space; the plotted component volatilities therefore combine by the sum of their squares. Panel C reports abnormal shock size relative to each category's own rolling history. Panel D shows the share of total smoothed variance attributable to the aggregate/common shock.

In [ ]:
plot_data = volatility_output.set_index('date').dropna(subset=['shock_volatility_ewma'])
ink = '#17212b'
grid = '#d8dee6'
total_color = '#1f5a93'
aggregate_color = '#d95f02'
dispersion_color = '#2a9d8f'
standardized_color = '#7b4ab5'
raw_color = '#9aa5b1'

fig, axes = plt.subplots(4, 1, figsize=(13.0, 11.0), sharex=True, gridspec_kw={'height_ratios': [1.0, 1.0, 0.85, 0.8], 'hspace': 0.25})

axes[0].plot(plot_data.index, plot_data['shock_intensity'], color=raw_color, lw=0.75, alpha=0.55, label='Monthly shock intensity')
axes[0].plot(plot_data.index, plot_data['shock_volatility_ewma'], color=total_color, lw=1.6, label='EWMA shock volatility')
axes[0].set_ylabel('Percentage points')
axes[0].set_title('A. Total category-level shock volatility', loc='left', fontsize=11.5, fontweight='bold')
axes[0].legend(loc='upper left', frameon=False, ncol=2, fontsize=8.8)

axes[1].plot(plot_data.index, plot_data['shock_volatility_ewma'], color=total_color, lw=1.5, label='Total')
axes[1].plot(plot_data.index, plot_data['aggregate_volatility_ewma'], color=aggregate_color, lw=1.25, label='Aggregate/common component')
axes[1].plot(plot_data.index, plot_data['dispersion_volatility_ewma'], color=dispersion_color, lw=1.25, label='Cross-category dispersion')
axes[1].set_ylabel('Percentage points')
axes[1].set_title('B. Volatility decomposition (squared components add)', loc='left', fontsize=11.5, fontweight='bold')
axes[1].legend(loc='upper left', frameon=False, ncol=3, fontsize=8.8)

axes[2].plot(plot_data.index, plot_data['standardized_shock_volatility_ewma'], color=standardized_color, lw=1.45)
axes[2].axhline(1, color=grid, lw=0.9, ls='--')
axes[2].set_ylabel('Historical-scale units')
axes[2].set_title('C. Standardized shock volatility', loc='left', fontsize=11.5, fontweight='bold')

axes[3].fill_between(plot_data.index, 0, plot_data['common_variance_share'] * 100, color=aggregate_color, alpha=0.35)
axes[3].plot(plot_data.index, plot_data['common_variance_share'] * 100, color=aggregate_color, lw=1.1)
share_axis_top = max(20, min(100, np.ceil(plot_data['common_variance_share'].max() * 10) * 10))
axes[3].set_ylim(0, share_axis_top)
axes[3].set_ylabel('Percent')
axes[3].set_title('D. Aggregate/common share of total shock variance', loc='left', fontsize=11.5, fontweight='bold')

for axis in axes:
    axis.grid(axis='y', color=grid, lw=0.7)
    axis.spines[['top', 'right', 'left']].set_visible(False)
    axis.spines['bottom'].set_color(grid)
    axis.tick_params(axis='both', colors=ink, labelsize=9)
    axis.margins(x=0)
axes[3].xaxis.set_major_locator(mdates.YearLocator(8))
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[3].set_xlim(plot_data.index[0], plot_data.index[-1] + pd.DateOffset(months=20))

fig.suptitle('Core PCE Shock Volatility', x=0.085, y=0.988, ha='left', fontsize=17, fontweight='bold', color=ink)
fig.text(0.085, 0.961, f'117-category core basket · {WINDOW}-month rolling AR({AR_LAGS}) · EWMA half-life {EWMA_HALFLIFE} months', ha='left', fontsize=10, color='#4f5965')
fig.text(0.085, 0.012, 'Source: BEA NIPA Underlying Detail tables U20404 and U20405; author calculations. Current-vintage historical reconstruction.', ha='left', fontsize=8.4, color='#4f5965')
fig.subplots_adjust(left=0.085, right=0.97, top=0.925, bottom=0.075)
fig.savefig(FIGURE_PNG, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(FIGURE_PNG)

## 8. CSV exports and validation summary

The monthly file contains raw variance inputs, smoothed volatility measures, the standardized measure, diagnostics, and coverage. The long file retains every valid category-month shock and its weight so results are auditable. No credential is included in either output.

In [ ]:
volatility_output.to_csv(INDEX_CSV, index=False, date_format='%Y-%m', float_format='%.15g')

shock_parts = []
for line in LINES:
    line_frame = pd.DataFrame({
        'date': infl.index,
        'line': line,
        'description': LINE_DESCRIPTION.loc[line],
        'residual': residual[line].to_numpy(),
        'residual_scale': residual_scale[line].to_numpy(),
        'standardized_residual': standardized_residual[line].to_numpy(),
        'spending': spend[line].to_numpy(),
        'weight': weights[line].to_numpy(),
        'standardized_weight': standardized_weights[line].to_numpy(),
    })
    shock_parts.append(line_frame.loc[raw_valid[line].to_numpy()])
category_shocks_output = pd.concat(shock_parts, ignore_index=True).sort_values(['date', 'line']).reset_index(drop=True)
category_shocks_output.to_csv(SHOCKS_CSV, index=False, date_format='%Y-%m', float_format='%.15g')

latest = volatility_output.dropna(subset=['shock_volatility_ewma']).iloc[-1]
runtime_seconds = time.perf_counter() - run_started
summary = pd.Series({
    'core_line_count': len(LINES),
    'first_raw_shock_month': volatility_output['date'].iloc[0].strftime('%Y-%m'),
    'first_smoothed_month': volatility_output.dropna(subset=['shock_volatility_ewma'])['date'].iloc[0].strftime('%Y-%m'),
    'latest_month': latest['date'].strftime('%Y-%m'),
    'latest_shock_volatility_ewma': float(latest['shock_volatility_ewma']),
    'latest_aggregate_volatility_ewma': float(latest['aggregate_volatility_ewma']),
    'latest_dispersion_volatility_ewma': float(latest['dispersion_volatility_ewma']),
    'latest_standardized_volatility_ewma': float(latest['standardized_shock_volatility_ewma']),
    'latest_common_variance_share': float(latest['common_variance_share']),
    'minimum_published_lines': int(volatility_output['n_valid'].min()),
    'minimum_weight_coverage': float(volatility_output['weight_coverage'].min()),
    'maximum_raw_decomposition_error': float(decomposition_error.abs().max()),
    'maximum_smoothed_decomposition_error': float(smoothed_decomposition_error.abs().max()),
    'description_mismatch_count': len(description_mismatches),
    'nonpositive_spending_observations': nonpositive_spend,
    'runtime_seconds': round(runtime_seconds, 2),
    'monthly_csv': str(INDEX_CSV),
    'category_csv': str(SHOCKS_CSV),
    'figure_png': str(FIGURE_PNG),
})
print(f'Wrote {len(volatility_output):,} monthly rows to {INDEX_CSV}')
print(f'Wrote {len(category_shocks_output):,} category-month rows to {SHOCKS_CSV}')
display(summary)

## 9. Interpretation and robustness

- `shock_volatility_ewma` is the headline measure of total category-level surprise magnitude. Opposite-signed shocks do not cancel.
- `aggregate_volatility_ewma` measures the magnitude of the weighted-average shock, where offsetting category shocks do cancel.
- `dispersion_volatility_ewma` measures heterogeneous relative-price shocks across core categories.
- `standardized_shock_volatility_ewma` asks whether shocks are large relative to each category's own rolling residual scale. Its reference value near one is descriptive rather than an exact statistical threshold.
- `shock_volatility_trailing12` provides an easily explained alternative to EWMA smoothing.

The squared aggregate and dispersion component volatilities sum to squared total volatility. Changing `AR_LAGS`, `EWMA_HALFLIFE`, or `TRAILING_WINDOW` changes the output tag or should be accompanied by a new tag. AR(3) is the principal model-robustness comparison. The results use current-vintage BEA data and are not a real-time-vintage reconstruction.